# Agent Evaluation & Testing — Measuring Agent Quality

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/26_agent_evaluation_and_testing.ipynb)

## What This Notebook Teaches You

You can't improve what you can't measure. Agents are complex systems — they use LLMs, tools, memory, and planning. How do you know if your agent is *good*? How do you detect regressions when you change something?

This notebook builds a **complete evaluation framework** for LLM-based agents:

1. **Metrics** — task completion, tool accuracy, reasoning quality, cost efficiency
2. **Golden datasets** — hand-crafted test suites with expected answers
3. **Automated scoring** — exact match, fuzzy match, and LLM-as-judge
4. **Evaluation harness** — run agents on test suites, aggregate results
5. **Cost tracking** — tokens, latency, and dollar estimates per task
6. **Regression testing** — save baselines and compare after changes

> **Research foundation**: LLM-as-judge evaluation follows Zheng et al. (2023), *"Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena."*

---

> **Prerequisites**: Modules 1–5 (especially the ReAct agent from Module 3).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~45–60 minutes.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What This Notebook Teaches You**
- Understand **: Environment Setup**
- Understand **Why Evaluate Agents?**
- Understand **Defining Evaluation Metrics**
- Understand **Building a Golden Dataset**


## Part 0: Environment Setup

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Evaluate Agents?

### The Measurement Problem

Traditional software has clear correctness criteria: unit tests pass or fail. Agents are different:

- **Non-deterministic**: Same input → different outputs each run
- **Multi-step**: Failure could occur at reasoning, tool use, or synthesis
- **Subjective**: "Good" answers require judgment, not just string matching

### What We Need to Measure

| Dimension | Question | Metric |
|-----------|----------|--------|
| **Correctness** | Did the agent get the right answer? | Task completion rate |
| **Tool usage** | Did it use the right tools correctly? | Tool accuracy |
| **Reasoning** | Was the thought process sound? | Reasoning quality score |
| **Efficiency** | How many tokens / how much time? | Cost per task |
| **Robustness** | Does it handle edge cases? | Error recovery rate |

Let's build infrastructure to measure all of these.

## 2. Defining Evaluation Metrics

We define a structured way to capture multi-dimensional evaluation results.

In [ ]:
@dataclass
class EvalMetrics:
    """Multi-dimensional evaluation result for a single task."""
    task_id: str
    task_description: str
    expected_answer: str
    actual_answer: str
    # Scores (0.0 to 1.0)
    exact_match: float = 0.0
    fuzzy_score: float = 0.0
    judge_score: float = 0.0
    # Tool usage
    tools_used: List[str] = field(default_factory=list)
    tools_expected: List[str] = field(default_factory=list)
    tool_accuracy: float = 0.0
    # Cost
    input_tokens: int = 0
    output_tokens: int = 0
    total_tokens: int = 0
    latency_seconds: float = 0.0
    num_steps: int = 0
    # Status
    completed: bool = False
    error: Optional[str] = None

    @property
    def composite_score(self) -> float:
        """Weighted composite across all dimensions."""
        return (
            0.3 * self.exact_match +
            0.2 * self.fuzzy_score +
            0.3 * self.judge_score +
            0.2 * self.tool_accuracy
        )

    def summary(self) -> str:
        return (
            f"Task: {self.task_id} | Composite: {self.composite_score:.2f} | "
            f"Exact: {self.exact_match:.0%} | Fuzzy: {self.fuzzy_score:.0%} | "
            f"Judge: {self.judge_score:.0%} | Tools: {self.tool_accuracy:.0%} | "
            f"Tokens: {self.total_tokens} | Steps: {self.num_steps}"
        )

# Quick test
m = EvalMetrics(
    task_id="test-001", task_description="Test",
    expected_answer="42", actual_answer="42",
    exact_match=1.0, fuzzy_score=1.0, judge_score=0.8, tool_accuracy=1.0
)
print(m.summary())
print(f"Composite score: {m.composite_score:.3f}")

## 3. Building a Golden Dataset

A golden dataset is a curated set of tasks with **known correct answers**. This is your ground truth for evaluation.

Key properties of a good golden dataset:
- **Diverse** — covers different task types (factual, reasoning, tool-use, multi-step)
- **Specific** — expected answers are clear and unambiguous
- **Realistic** — tasks reflect actual agent use cases
- **Sized right** — large enough for statistical significance (15+ tasks)

In [ ]:
@dataclass
class TestCase:
    """A single test case in the golden dataset."""
    task_id: str
    question: str
    expected_answer: str
    expected_tools: List[str] = field(default_factory=list)
    category: str = "general"
    difficulty: str = "medium"  # easy, medium, hard

# Build our golden dataset of 15 tasks
GOLDEN_DATASET = [
    TestCase(
        task_id="math-001",
        question="What is 347 * 23?",
        expected_answer="7981",
        expected_tools=["calculator"],
        category="math",
        difficulty="easy"
    ),
    TestCase(
        task_id="math-002",
        question="What is the square root of 1764?",
        expected_answer="42",
        expected_tools=["calculator"],
        category="math",
        difficulty="easy"
    ),
    TestCase(
        task_id="math-003",
        question="If a rectangle has area 120 and width 8, what is its length?",
        expected_answer="15",
        expected_tools=["calculator"],
        category="math",
        difficulty="medium"
    ),
    TestCase(
        task_id="fact-001",
        question="What is the capital of France?",
        expected_answer="Paris",
        expected_tools=["knowledge_base"],
        category="factual",
        difficulty="easy"
    ),
    TestCase(
        task_id="fact-002",
        question="What is the chemical symbol for gold?",
        expected_answer="Au",
        expected_tools=["knowledge_base"],
        category="factual",
        difficulty="easy"
    ),
    TestCase(
        task_id="fact-003",
        question="Who wrote the novel '1984'?",
        expected_answer="George Orwell",
        expected_tools=["knowledge_base"],
        category="factual",
        difficulty="easy"
    ),
    TestCase(
        task_id="reason-001",
        question="If all roses are flowers and some flowers are red, can we conclude all roses are red?",
        expected_answer="No",
        expected_tools=[],
        category="reasoning",
        difficulty="medium"
    ),
    TestCase(
        task_id="reason-002",
        question="A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much does the ball cost?",
        expected_answer="$0.05",
        expected_tools=["calculator"],
        category="reasoning",
        difficulty="medium"
    ),
    TestCase(
        task_id="reason-003",
        question="If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
        expected_answer="5 minutes",
        expected_tools=["calculator"],
        category="reasoning",
        difficulty="hard"
    ),
    TestCase(
        task_id="multi-001",
        question="Calculate 15% tip on a $84.50 restaurant bill and round to the nearest cent.",
        expected_answer="$12.68",
        expected_tools=["calculator"],
        category="multi-step",
        difficulty="medium"
    ),
    TestCase(
        task_id="multi-002",
        question="Convert 72 degrees Fahrenheit to Celsius. Use the formula C = (F - 32) * 5/9.",
        expected_answer="22.22",
        expected_tools=["calculator"],
        category="multi-step",
        difficulty="medium"
    ),
    TestCase(
        task_id="multi-003",
        question="A store has a 25% off sale. An item costs $60 before discount. What is the sale price after 8% tax?",
        expected_answer="$48.60",
        expected_tools=["calculator"],
        category="multi-step",
        difficulty="hard"
    ),
    TestCase(
        task_id="lookup-001",
        question="Look up the population of Tokyo and report whether it exceeds 10 million.",
        expected_answer="Yes",
        expected_tools=["knowledge_base"],
        category="lookup",
        difficulty="medium"
    ),
    TestCase(
        task_id="lookup-002",
        question="What is the boiling point of water in Fahrenheit?",
        expected_answer="212",
        expected_tools=["knowledge_base"],
        category="lookup",
        difficulty="easy"
    ),
    TestCase(
        task_id="lookup-003",
        question="How many planets are in our solar system?",
        expected_answer="8",
        expected_tools=["knowledge_base"],
        category="factual",
        difficulty="easy"
    ),
]

# Dataset statistics
categories = defaultdict(int)
difficulties = defaultdict(int)
for tc in GOLDEN_DATASET:
    categories[tc.category] += 1
    difficulties[tc.difficulty] += 1

print(f"Golden dataset: {len(GOLDEN_DATASET)} test cases")
print(f"\nCategories: {dict(categories)}")
print(f"Difficulties: {dict(difficulties)}")
print(f"\nSample task:")
print(f"  ID: {GOLDEN_DATASET[0].task_id}")
print(f"  Q:  {GOLDEN_DATASET[0].question}")
print(f"  A:  {GOLDEN_DATASET[0].expected_answer}")
print(f"  Tools: {GOLDEN_DATASET[0].expected_tools}")

## 4. Automated Scoring Functions

We build three levels of scoring, from simple to sophisticated:

| Scorer | How it works | When to use |
|--------|-------------|-------------|
| **Exact match** | Normalized string comparison | Factual, numeric answers |
| **Fuzzy match** | Keyword overlap ratio | Open-ended, multi-word answers |
| **LLM-as-judge** | Another LLM rates quality 1–5 | Complex, subjective answers |

In [ ]:
def normalize_answer(text: str) -> str:
    """Normalize answer for comparison: lowercase, strip, remove punctuation."""
    text = text.lower().strip()
    # Remove common prefixes
    for prefix in ["the answer is", "answer:", "result:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    # Remove $, commas, trailing periods
    text = text.replace("$", "").replace(",", "").rstrip(".")
    return text

def score_exact_match(expected: str, actual: str) -> float:
    """Binary exact match after normalization."""
    return 1.0 if normalize_answer(expected) == normalize_answer(actual) else 0.0

# Test exact match scoring
test_pairs = [
    ("42", "42", 1.0),
    ("42", "The answer is 42", 1.0),
    ("Paris", "paris", 1.0),
    ("$12.68", "12.68", 1.0),
    ("42", "43", 0.0),
    ("Yes", "No", 0.0),
]

print("Exact Match Scoring Tests:")
print("-" * 50)
for expected, actual, want in test_pairs:
    got = score_exact_match(expected, actual)
    status = "✓" if got == want else "✗"
    print(f"  {status} expected='{expected}' actual='{actual}' → {got:.0f} (want {want:.0f})")

In [ ]:
def score_fuzzy_match(expected: str, actual: str) -> float:
    """Keyword overlap ratio between expected and actual answers.

    Computes: |keywords_expected ∩ keywords_actual| / |keywords_expected|
    This measures recall — did the actual answer contain the key terms?
    """
    def extract_keywords(text: str) -> set:
        text = text.lower()
        # Remove punctuation and split
        words = re.findall(r'\b\w+\b', text)
        # Remove stop words
        stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'of', 'in',
                      'to', 'for', 'and', 'or', 'it', 'that', 'this', 'be'}
        return {w for w in words if w not in stop_words}

    expected_kw = extract_keywords(expected)
    actual_kw = extract_keywords(actual)

    if not expected_kw:
        return 1.0 if not actual_kw else 0.0

    overlap = expected_kw & actual_kw
    return len(overlap) / len(expected_kw)

# Test fuzzy scoring
fuzzy_tests = [
    ("George Orwell", "The author is George Orwell", 1.0),
    ("George Orwell", "Orwell wrote it", 0.5),
    ("George Orwell", "Charles Dickens", 0.0),
    ("5 minutes", "It would take 5 minutes", 1.0),
]

print("Fuzzy Match Scoring Tests:")
print("-" * 50)
for expected, actual, want in fuzzy_tests:
    got = score_fuzzy_match(expected, actual)
    status = "✓" if abs(got - want) < 0.01 else "≈"
    print(f"  {status} expected='{expected}' actual='{actual}' → {got:.2f} (want {want:.2f})")

In [ ]:
def score_with_llm_judge(question: str, expected: str, actual: str) -> Tuple[float, str]:
    """Use the LLM itself as a judge to rate answer quality on a 1-5 scale.

    Based on Zheng et al. 2023 'Judging LLM-as-a-Judge' methodology.
    Returns (normalized_score, explanation).
    """
    judge_prompt = f"""You are an impartial judge evaluating an AI assistant's answer.

Question: {question}
Reference answer: {expected}
Assistant's answer: {actual}

Rate the assistant's answer on a scale of 1-5:
1 = Completely wrong or irrelevant
2 = Partially addresses the question but has major errors
3 = Addresses the question but with minor errors or missing details
4 = Correct and relevant, minor issues only
5 = Perfect — accurate, complete, and well-expressed

Respond with EXACTLY this format:
Score: [1-5]
Reason: [one sentence explanation]"""

    messages = [{"role": "user", "content": judge_prompt}]
    response = generate(messages, max_new_tokens=100, temperature=0.1, do_sample=True)

    # Parse score from response
    score = 3  # default
    reason = response.strip()
    score_match = re.search(r'Score:\s*(\d)', response)
    if score_match:
        score = int(score_match.group(1))
        score = max(1, min(5, score))

    reason_match = re.search(r'Reason:\s*(.+)', response, re.IGNORECASE)
    if reason_match:
        reason = reason_match.group(1).strip()

    # Normalize to 0-1 range: (score - 1) / 4
    normalized = (score - 1) / 4.0
    return normalized, reason

# Test the LLM judge on a few examples
print("LLM-as-Judge Scoring:")
print("=" * 60)

judge_tests = [
    ("What is 347 * 23?", "7981", "7981"),
    ("What is 347 * 23?", "7981", "About 8000"),
    ("What is the capital of France?", "Paris", "The capital of France is Paris, known as the City of Light."),
    ("What is the capital of France?", "Paris", "London"),
]

for q, expected, actual in judge_tests:
    score, reason = score_with_llm_judge(q, expected, actual)
    print(f"\n  Q: {q}")
    print(f"  Expected: {expected}")
    print(f"  Actual:   {actual}")
    print(f"  Judge score: {score:.2f} ({score*4+1:.0f}/5)")
    print(f"  Reason: {reason}")

## 5. Tool Usage Accuracy

Beyond answer correctness, we evaluate whether the agent used the **right tools** for the job.

In [ ]:
def score_tool_accuracy(expected_tools: List[str], actual_tools: List[str]) -> float:
    """Score tool usage accuracy.

    Measures both precision and recall of tool usage:
    - Did the agent use the tools it should have? (recall)
    - Did it avoid unnecessary tools? (precision)
    Returns F1 score of tool sets.
    """
    if not expected_tools and not actual_tools:
        return 1.0  # No tools expected, none used — perfect
    if not expected_tools and actual_tools:
        return 0.5  # Used tools when none needed — partial penalty
    if expected_tools and not actual_tools:
        return 0.0  # Needed tools but used none

    expected_set = set(expected_tools)
    actual_set = set(actual_tools)

    true_positive = len(expected_set & actual_set)
    precision = true_positive / len(actual_set) if actual_set else 0.0
    recall = true_positive / len(expected_set) if expected_set else 0.0

    if precision + recall == 0:
        return 0.0
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

# Test tool accuracy scoring
tool_tests = [
    (["calculator"], ["calculator"], 1.0),
    (["calculator", "knowledge_base"], ["calculator", "knowledge_base"], 1.0),
    (["calculator"], ["calculator", "web_search"], 0.67),
    (["calculator"], [], 0.0),
    ([], [], 1.0),
]

print("Tool Accuracy Scoring:")
print("-" * 50)
for expected, actual, want in tool_tests:
    got = score_tool_accuracy(expected, actual)
    status = "✓" if abs(got - want) < 0.05 else "≈"
    print(f"  {status} expected={expected} actual={actual} → {got:.2f} (want ~{want:.2f})")

## 6. The AgentEvaluator — Full Evaluation Harness

Now we combine everything into a class that runs an agent against the golden dataset and produces a comprehensive evaluation report.

In [ ]:
class AgentEvaluator:
    """Evaluation harness for LLM-based agents."""

    def __init__(self, agent_fn: Callable, name: str = "Agent"):
        """
        Args:
            agent_fn: Function(question: str) -> dict with keys:
                'answer': str, 'tools_used': List[str], 'steps': int
            name: Name for this agent in reports.
        """
        self.agent_fn = agent_fn
        self.name = name
        self.results: List[EvalMetrics] = []

    def evaluate_single(self, test_case: TestCase, use_judge: bool = True) -> EvalMetrics:
        """Evaluate agent on a single test case."""
        metrics = EvalMetrics(
            task_id=test_case.task_id,
            task_description=test_case.question,
            expected_answer=test_case.expected_answer,
            actual_answer="",
        )

        start_time = time.time()
        try:
            result = self.agent_fn(test_case.question)
            metrics.actual_answer = result.get("answer", "")
            metrics.tools_used = result.get("tools_used", [])
            metrics.num_steps = result.get("steps", 0)
            metrics.input_tokens = result.get("input_tokens", 0)
            metrics.output_tokens = result.get("output_tokens", 0)
            metrics.total_tokens = metrics.input_tokens + metrics.output_tokens
            metrics.completed = True
        except Exception as e:
            metrics.error = str(e)
            metrics.completed = False

        metrics.latency_seconds = time.time() - start_time

        # Score the result
        if metrics.completed:
            metrics.exact_match = score_exact_match(
                test_case.expected_answer, metrics.actual_answer
            )
            metrics.fuzzy_score = score_fuzzy_match(
                test_case.expected_answer, metrics.actual_answer
            )
            metrics.tool_accuracy = score_tool_accuracy(
                test_case.expected_tools, metrics.tools_used
            )
            if use_judge:
                metrics.judge_score, _ = score_with_llm_judge(
                    test_case.question, test_case.expected_answer, metrics.actual_answer
                )

        return metrics

    def evaluate_dataset(self, dataset: List[TestCase], use_judge: bool = True) -> List[EvalMetrics]:
        """Evaluate agent on a full dataset."""
        self.results = []
        for i, tc in enumerate(dataset):
            print(f"  [{i+1}/{len(dataset)}] {tc.task_id}: {tc.question[:50]}...")
            metrics = self.evaluate_single(tc, use_judge=use_judge)
            self.results.append(metrics)
            print(f"    → {metrics.summary()}")
        return self.results

    def aggregate_report(self) -> Dict[str, Any]:
        """Produce aggregate statistics from evaluation results."""
        if not self.results:
            return {"error": "No results to aggregate"}

        n = len(self.results)
        completed = [r for r in self.results if r.completed]

        report = {
            "agent_name": self.name,
            "total_tasks": n,
            "completed": len(completed),
            "completion_rate": len(completed) / n,
            "avg_exact_match": sum(r.exact_match for r in completed) / max(len(completed), 1),
            "avg_fuzzy_score": sum(r.fuzzy_score for r in completed) / max(len(completed), 1),
            "avg_judge_score": sum(r.judge_score for r in completed) / max(len(completed), 1),
            "avg_tool_accuracy": sum(r.tool_accuracy for r in completed) / max(len(completed), 1),
            "avg_composite": sum(r.composite_score for r in completed) / max(len(completed), 1),
            "total_tokens": sum(r.total_tokens for r in completed),
            "avg_tokens_per_task": sum(r.total_tokens for r in completed) / max(len(completed), 1),
            "avg_latency": sum(r.latency_seconds for r in completed) / max(len(completed), 1),
            "avg_steps": sum(r.num_steps for r in completed) / max(len(completed), 1),
        }

        # Per-category breakdown
        categories = defaultdict(list)
        for r in completed:
            # Look up category from the dataset
            for tc in GOLDEN_DATASET:
                if tc.task_id == r.task_id:
                    categories[tc.category].append(r)
                    break

        report["by_category"] = {}
        for cat, results in categories.items():
            report["by_category"][cat] = {
                "count": len(results),
                "avg_composite": sum(r.composite_score for r in results) / len(results),
                "avg_exact_match": sum(r.exact_match for r in results) / len(results),
            }

        return report

    def print_report(self):
        """Pretty-print the aggregate report."""
        report = self.aggregate_report()
        print(f"\n{'='*60}")
        print(f"  EVALUATION REPORT: {report['agent_name']}")
        print(f"{'='*60}")
        print(f"  Tasks: {report['completed']}/{report['total_tasks']} completed ({report['completion_rate']:.0%})")
        print(f"\n  Scores:")
        print(f"    Exact Match:    {report['avg_exact_match']:.2%}")
        print(f"    Fuzzy Match:    {report['avg_fuzzy_score']:.2%}")
        print(f"    LLM Judge:      {report['avg_judge_score']:.2%}")
        print(f"    Tool Accuracy:  {report['avg_tool_accuracy']:.2%}")
        print(f"    ─────────────────────────")
        print(f"    Composite:      {report['avg_composite']:.2%}")
        print(f"\n  Cost:")
        print(f"    Total tokens:   {report['total_tokens']:,}")
        print(f"    Avg per task:   {report['avg_tokens_per_task']:,.0f} tokens")
        print(f"    Avg latency:    {report['avg_latency']:.1f}s")
        print(f"    Avg steps:      {report['avg_steps']:.1f}")
        if report.get("by_category"):
            print(f"\n  By Category:")
            for cat, stats in report["by_category"].items():
                print(f"    {cat}: {stats['avg_composite']:.2%} composite ({stats['count']} tasks)")
        print(f"{'='*60}")

print("✓ AgentEvaluator class defined")

## 7. Build a ReAct Agent to Evaluate

We need an agent to test. We'll build a simple ReAct agent with tools, then run it through our evaluator.

In [ ]:
# Define tools for the agent
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        # Safe eval for math
        allowed = {
            'abs': abs, 'round': round, 'min': min, 'max': max,
            'pow': pow, 'sqrt': math.sqrt, 'pi': math.pi, 'e': math.e,
        }
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def knowledge_base(query: str) -> str:
    """Look up facts from a simple knowledge base."""
    kb = {
        "capital of france": "Paris is the capital of France.",
        "chemical symbol for gold": "The chemical symbol for gold is Au.",
        "who wrote 1984": "George Orwell wrote the novel '1984'.",
        "population of tokyo": "Tokyo has a population of approximately 13.96 million (metro: 37 million).",
        "boiling point of water": "The boiling point of water is 100°C (212°F) at sea level.",
        "planets in solar system": "There are 8 planets in our solar system.",
    }
    query_lower = query.lower()
    for key, value in kb.items():
        if key in query_lower or any(word in query_lower for word in key.split()):
            return value
    return f"No information found for: {query}"

TOOLS = {
    "calculator": {
        "fn": calculator,
        "description": "Evaluate a mathematical expression. Input: math expression string.",
    },
    "knowledge_base": {
        "fn": knowledge_base,
        "description": "Look up factual information. Input: search query string.",
    },
}

print("✓ Tools defined:")
for name, tool in TOOLS.items():
    print(f"  - {name}: {tool['description']}")

In [ ]:
def react_agent(question: str, max_steps: int = 5) -> dict:
    """A simple ReAct agent that reasons and acts with tools.

    Returns dict with: answer, tools_used, steps, input_tokens, output_tokens.
    """
    tools_desc = "\n".join(
        f"- {name}: {info['description']}" for name, info in TOOLS.items()
    )

    system_prompt = f"""You are a helpful assistant with access to these tools:
{tools_desc}

To use a tool, respond with EXACTLY:
Thought: [your reasoning]
Action: [tool_name]
Action Input: [input to tool]

When you have the final answer, respond with EXACTLY:
Thought: [your reasoning]
Final Answer: [your answer]

Always use tools when they would help. Give concise final answers."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    tools_used = []
    total_input_tokens = 0
    total_output_tokens = 0

    for step in range(max_steps):
        response = generate(messages, max_new_tokens=256, temperature=0.1, do_sample=True)

        # Estimate tokens (approximate)
        input_text = " ".join(m["content"] for m in messages)
        total_input_tokens += len(input_text.split()) * 1.3  # rough estimate
        total_output_tokens += len(response.split()) * 1.3

        # Check for Final Answer
        final_match = re.search(r'Final Answer:\s*(.+)', response, re.IGNORECASE)
        if final_match:
            return {
                "answer": final_match.group(1).strip(),
                "tools_used": tools_used,
                "steps": step + 1,
                "input_tokens": int(total_input_tokens),
                "output_tokens": int(total_output_tokens),
            }

        # Check for Action
        action_match = re.search(r'Action:\s*(\w+)', response)
        input_match = re.search(r'Action Input:\s*(.+)', response)

        if action_match and input_match:
            tool_name = action_match.group(1).strip()
            tool_input = input_match.group(1).strip()

            if tool_name in TOOLS:
                tools_used.append(tool_name)
                result = TOOLS[tool_name]["fn"](tool_input)
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"Observation: {result}"})
            else:
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"Observation: Tool '{tool_name}' not found. Available: {list(TOOLS.keys())}"})
        else:
            # Model didn't follow format — treat response as final answer
            return {
                "answer": response.strip().split("\n")[0],
                "tools_used": tools_used,
                "steps": step + 1,
                "input_tokens": int(total_input_tokens),
                "output_tokens": int(total_output_tokens),
            }

    return {
        "answer": "Max steps reached without final answer.",
        "tools_used": tools_used,
        "steps": max_steps,
        "input_tokens": int(total_input_tokens),
        "output_tokens": int(total_output_tokens),
    }

# Quick test
result = react_agent("What is 347 * 23?")
print(f"Answer: {result['answer']}")
print(f"Tools used: {result['tools_used']}")
print(f"Steps: {result['steps']}")

## 8. Evaluate the ReAct Agent

Now we run our agent through the full evaluation harness on 10 test cases. This gives us a realistic picture of agent quality across dimensions.

In [ ]:
# Select 10 test cases for evaluation (mix of categories)
eval_subset = GOLDEN_DATASET[:10]

print(f"Evaluating ReAct agent on {len(eval_subset)} test cases...")
print(f"Categories: {dict(defaultdict(int, {tc.category: 1 for tc in eval_subset}))}")
print("-" * 60)

evaluator = AgentEvaluator(react_agent, name="ReAct-Qwen3-8B")
results = evaluator.evaluate_dataset(eval_subset, use_judge=True)

# Print the full report
evaluator.print_report()

## 9. Cost Tracking & Efficiency Analysis

For production agents, cost matters. Let's analyze the cost profile of our agent.

In [ ]:
class CostTracker:
    """Track and estimate costs for agent operations."""

    # Approximate pricing (per 1M tokens)
    # For open-source models running on Colab, we estimate GPU-hour cost
    COST_PER_1M_INPUT_TOKENS = 0.15   # estimate for Colab T4
    COST_PER_1M_OUTPUT_TOKENS = 0.60  # output is more expensive (slower)

    def __init__(self):
        self.tasks: List[Dict] = []

    def record(self, task_id: str, input_tokens: int, output_tokens: int,
               latency: float, steps: int):
        self.tasks.append({
            "task_id": task_id,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
            "latency": latency,
            "steps": steps,
            "estimated_cost": (
                input_tokens * self.COST_PER_1M_INPUT_TOKENS / 1_000_000 +
                output_tokens * self.COST_PER_1M_OUTPUT_TOKENS / 1_000_000
            ),
        })

    def summary(self) -> Dict:
        if not self.tasks:
            return {}
        total_input = sum(t["input_tokens"] for t in self.tasks)
        total_output = sum(t["output_tokens"] for t in self.tasks)
        total_cost = sum(t["estimated_cost"] for t in self.tasks)
        total_time = sum(t["latency"] for t in self.tasks)
        n = len(self.tasks)
        return {
            "num_tasks": n,
            "total_tokens": total_input + total_output,
            "avg_tokens_per_task": (total_input + total_output) / n,
            "total_estimated_cost": total_cost,
            "avg_cost_per_task": total_cost / n,
            "total_time": total_time,
            "avg_latency": total_time / n,
            "tokens_per_second": (total_input + total_output) / max(total_time, 0.01),
        }

    def print_summary(self):
        s = self.summary()
        print(f"\n{'='*50}")
        print(f"  COST ANALYSIS ({s['num_tasks']} tasks)")
        print(f"{'='*50}")
        print(f"  Total tokens:      {s['total_tokens']:,}")
        print(f"  Avg tokens/task:   {s['avg_tokens_per_task']:,.0f}")
        print(f"  Estimated cost:    ${s['total_estimated_cost']:.4f}")
        print(f"  Avg cost/task:     ${s['avg_cost_per_task']:.6f}")
        print(f"  Total time:        {s['total_time']:.1f}s")
        print(f"  Avg latency:       {s['avg_latency']:.1f}s")
        print(f"  Throughput:        {s['tokens_per_second']:.0f} tokens/s")
        print(f"{'='*50}")

# Build cost report from evaluation results
tracker = CostTracker()
for r in evaluator.results:
    tracker.record(r.task_id, r.input_tokens, r.output_tokens,
                   r.latency_seconds, r.num_steps)

tracker.print_summary()

# Per-task cost breakdown
print("\nPer-task breakdown:")
print(f"  {'Task':<12} {'Tokens':>8} {'Cost':>10} {'Time':>8} {'Steps':>6}")
print(f"  {'-'*12} {'-'*8} {'-'*10} {'-'*8} {'-'*6}")
for t in tracker.tasks:
    print(f"  {t['task_id']:<12} {t['total_tokens']:>8,} ${t['estimated_cost']:>9.6f} {t['latency']:.1f}s{' ':>4} {t['steps']:>5}")

## 10. Regression Testing — Baselines and Comparisons

When you change your agent (new prompts, different tools, model updates), you need to know if quality improved or degraded. Regression testing saves a baseline and compares new runs against it.

In [ ]:
class RegressionTracker:
    """Save evaluation baselines and detect regressions."""

    def __init__(self):
        self.baselines: Dict[str, Dict] = {}

    def save_baseline(self, name: str, report: Dict):
        """Save an evaluation report as a baseline."""
        self.baselines[name] = {
            "report": report,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        print(f"✓ Baseline saved: '{name}' at {self.baselines[name]['timestamp']}")

    def compare(self, name: str, new_report: Dict, threshold: float = 0.05) -> Dict:
        """Compare a new report against a saved baseline.

        Args:
            name: Baseline name to compare against.
            new_report: New evaluation report.
            threshold: Minimum change to flag as significant.

        Returns:
            Comparison dict with deltas and regression flags.
        """
        if name not in self.baselines:
            return {"error": f"No baseline named '{name}'"}

        baseline = self.baselines[name]["report"]
        comparison = {"baseline_name": name, "metrics": {}, "regressions": []}

        metrics_to_compare = [
            "avg_exact_match", "avg_fuzzy_score", "avg_judge_score",
            "avg_tool_accuracy", "avg_composite", "avg_tokens_per_task",
        ]

        for metric in metrics_to_compare:
            old_val = baseline.get(metric, 0)
            new_val = new_report.get(metric, 0)
            delta = new_val - old_val

            is_cost = metric in ["avg_tokens_per_task"]
            improved = delta < -threshold if is_cost else delta > threshold
            regressed = delta > threshold if is_cost else delta < -threshold

            comparison["metrics"][metric] = {
                "baseline": old_val,
                "current": new_val,
                "delta": delta,
                "status": "improved" if improved else ("regressed" if regressed else "stable"),
            }

            if regressed:
                comparison["regressions"].append(metric)

        return comparison

    def print_comparison(self, comparison: Dict):
        """Pretty-print a comparison report."""
        print(f"\n{'='*65}")
        print(f"  REGRESSION REPORT vs '{comparison['baseline_name']}'")
        print(f"{'='*65}")
        print(f"  {'Metric':<25} {'Baseline':>10} {'Current':>10} {'Delta':>8} {'Status':>10}")
        print(f"  {'-'*25} {'-'*10} {'-'*10} {'-'*8} {'-'*10}")

        for metric, data in comparison["metrics"].items():
            symbol = {"improved": "↑", "regressed": "↓", "stable": "="}.get(data["status"], "?")
            print(f"  {metric:<25} {data['baseline']:>10.4f} {data['current']:>10.4f} "
                  f"{data['delta']:>+8.4f} {symbol} {data['status']}")

        if comparison["regressions"]:
            print(f"\n  ⚠ REGRESSIONS DETECTED in: {', '.join(comparison['regressions'])}")
        else:
            print(f"\n  ✓ No regressions detected.")
        print(f"{'='*65}")

# Demo: Save baseline, simulate a change, compare
regression = RegressionTracker()

# Save current results as baseline
baseline_report = evaluator.aggregate_report()
regression.save_baseline("v1-react-agent", baseline_report)

# Simulate a "new version" with slightly different scores
simulated_report = {**baseline_report}
simulated_report["avg_exact_match"] = baseline_report["avg_exact_match"] - 0.1  # simulated regression
simulated_report["avg_composite"] = baseline_report["avg_composite"] - 0.08
simulated_report["avg_tokens_per_task"] = baseline_report["avg_tokens_per_task"] * 0.8  # improved cost

comparison = regression.compare("v1-react-agent", simulated_report)
regression.print_comparison(comparison)

## Exercises

**Exercise 1:** Design an eval dataset for a new agent use case with at least 10 test cases.

**Exercise 2:** Implement a custom metric that measures tool-use efficiency (calls per task).

**Exercise 3:** Run the evaluation on a different model and analyze how agent behavior changes.

## 11. LLM-as-Judge — Research and Best Practices

### Research Background: Zheng et al. 2023

The LLM-as-judge approach was systematically studied in *"Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena"* (Zheng et al., 2023). Key findings:

1. **Strong LLMs match human judges** — GPT-4-level models agree with human preferences >80% of the time
2. **Position bias** — judges tend to favor the first response in A/B comparisons. Mitigation: swap positions and average.
3. **Verbosity bias** — judges tend to favor longer responses. Mitigation: instruct judges to value conciseness.
4. **Self-enhancement bias** — models tend to rate their own outputs higher. Mitigation: use a different model as judge than the agent being evaluated.

### Limitations for Open-Source Models

Our Qwen3-8B judge has lower agreement with human judges than frontier models. For production, consider:
- Using a stronger model as judge (e.g., GPT-4 via API)
- Averaging multiple judge calls
- Combining LLM-judge with rule-based scoring (as we do with exact + fuzzy + judge)

Let's build an improved judge with bias mitigation.

In [ ]:
def improved_llm_judge(question: str, expected: str, actual: str,
                        num_samples: int = 2) -> Tuple[float, str]:
    """Improved LLM judge with bias mitigation.

    Mitigations:
    1. Explicit instruction to penalize verbosity
    2. Multiple samples averaged for stability
    3. Structured rubric for consistent scoring
    """
    rubric = """Scoring rubric:
1 = Wrong answer or completely irrelevant to the question
2 = Partially relevant but contains major factual errors
3 = Correct direction but incomplete or has minor errors
4 = Correct and complete, minor style issues only
5 = Perfect — accurate, concise, and directly answers the question

IMPORTANT: Prefer concise answers. Do NOT give higher scores just because an answer is longer."""

    judge_prompt = f"""{rubric}

Question: {question}
Reference answer: {expected}
Assistant's answer: {actual}

Rate the assistant's answer. Respond ONLY with:
Score: [1-5]
Reason: [one sentence]"""

    scores = []
    reasons = []

    for _ in range(num_samples):
        messages = [{"role": "user", "content": judge_prompt}]
        response = generate(messages, max_new_tokens=80, temperature=0.3, do_sample=True)

        score_match = re.search(r'Score:\s*(\d)', response)
        score = int(score_match.group(1)) if score_match else 3
        score = max(1, min(5, score))
        scores.append(score)

        reason_match = re.search(r'Reason:\s*(.+)', response)
        reasons.append(reason_match.group(1).strip() if reason_match else response.strip())

    avg_score = sum(scores) / len(scores)
    normalized = (avg_score - 1) / 4.0
    return normalized, f"Avg {avg_score:.1f}/5 over {num_samples} samples. {reasons[0]}"

# Compare basic vs improved judge
print("Comparing Basic vs Improved Judge:")
print("=" * 60)

test = ("What is 347 * 23?", "7981", "The answer is 7981.")
basic_score, basic_reason = score_with_llm_judge(*test)
improved_score, improved_reason = improved_llm_judge(*test)

print(f"  Q: {test[0]}")
print(f"  Expected: {test[1]} | Actual: {test[2]}")
print(f"  Basic judge:    {basic_score:.2f} — {basic_reason}")
print(f"  Improved judge: {improved_score:.2f} — {improved_reason}")

## 12. Key Takeaways

### What We Built

| Component | Purpose |
|-----------|---------|
| **EvalMetrics** | Multi-dimensional scoring dataclass |
| **Golden Dataset** | 15 curated test cases with expected answers |
| **Exact Match** | Normalized string comparison scorer |
| **Fuzzy Match** | Keyword overlap ratio scorer |
| **LLM-as-Judge** | LLM rates answer quality 1–5 (Zheng et al. 2023) |
| **Tool Accuracy** | F1 score on expected vs actual tool usage |
| **AgentEvaluator** | Full harness: run agent → score → aggregate |
| **CostTracker** | Token and latency tracking with cost estimates |
| **RegressionTracker** | Save baselines, detect quality regressions |

### Key Principles

1. **Multi-dimensional evaluation** — no single metric captures agent quality
2. **Combine automated + LLM scoring** — use exact/fuzzy for cheap bulk evaluation, LLM-judge for nuanced assessment
3. **Golden datasets must be curated** — garbage in, garbage out
4. **Track costs alongside quality** — the best agent might also be the most expensive
5. **Regression testing is essential** — every change needs comparison against baseline

### What's Next

In the next notebook, we'll use these evaluation tools to measure the impact of **cost and latency optimizations** — caching, prompt compression, and model routing.